In [68]:
import re
import json
import time
import random
import math
from pathlib import Path

import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [69]:
BASE_URL = "https://www.10000recipe.com"

LIST_URL = BASE_URL + "/recipe/list.html"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0.0.0 Safari/537.36"
    )
}

DATA_DIR = Path("./recipe_data")
DATA_DIR.mkdir(exist_ok=True)

LIST_FILE = DATA_DIR / "recipe_list.csv"
SELECTED_FILE = DATA_DIR / "selected_10000.csv"
DETAIL_FILE = DATA_DIR / "recipes_10000.jsonl"

In [70]:
session = requests.Session()

retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update(HEADERS)

In [71]:
def parse_views(text):
    if not text:
        return 0

    text = text.replace(",", "").strip()

    match = re.search(r"([\d.]+)\s*(만|천)?", text)

    if not match:
        return 0

    number = float(match.group(1))
    unit = match.group(2)

    if unit == "만":
        number *= 10_000

    elif unit == "천":
        number *= 1_000

    return int(number)

In [72]:
def parse_list_page(page):
    params = {
        "order": "date",   # 최신순
        "page": page
    }

    response = session.get(
        LIST_URL,
        params=params,
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # 현재 사이트 기본 구조
    items = soup.select("ul.common_sp_list_ul > li")

    # 혹시 class 구조가 조금 달라졌을 경우
    if not items:
        items = []

        for li in soup.select("li"):

            text = li.get_text(" ", strip=True)

            if "조회수" not in text:
                continue

            if li.select_one('a[href^="/recipe/"]'):
                items.append(li)

    recipes = []

    for rank, item in enumerate(items, start=1):

        # 제목
        title_element = item.select_one(
            ".common_sp_caption_tit"
        )

        if title_element:
            title = title_element.get_text(
                " ",
                strip=True
            )
        else:
            title = None

        # 레시피 링크
        recipe_link = item.select_one(
            'a[href^="/recipe/"]'
        )

        if recipe_link is None:
            continue

        href = recipe_link.get("href", "")

        id_match = re.search(
            r"/recipe/(\d+)",
            href
        )

        if not id_match:
            continue

        recipe_id = id_match.group(1)

        url = BASE_URL + f"/recipe/{recipe_id}"

        # 전체 텍스트에서 조회수 찾기
        item_text = item.get_text(
            " ",
            strip=True
        )

        view_match = re.search(
            r"조회수\s*([\d,.]+\s*(?:만|천)?)",
            item_text
        )

        views_raw = (
            view_match.group(1)
            if view_match
            else "0"
        )

        views = parse_views(views_raw)

        # 작성자
        author_element = item.select_one(
            ".common_sp_caption_rv_name"
        )

        author = (
            author_element.get_text(
                " ",
                strip=True
            )
            if author_element
            else None
        )

        recipes.append({
            "recipe_id": recipe_id,
            "title": title,
            "url": url,
            "views": views,
            "views_raw": views_raw,
            "author": author,
            "list_page": page,
            "page_rank": rank
        })

    return recipes

In [73]:
test_rows = parse_list_page(1)

len(test_rows)

40

In [74]:
pd.DataFrame(test_rows).head(10)

,recipe_id,title,url,views,views_raw,author,list_page,page_rank
0,7083207,오징어국 만드는 법 콩나물 듬뿍 넣은 얼큰한 해장국 레시피,https://www.10000recipe.com/recipe/7083207,0,0,춤추는루나,1,1
1,7083206,새송이버섯볶음 들깨가루 넣어 고소하게 10분 버섯볶음 레시피,https://www.10000recipe.com/recipe/7083206,2,2,춤추는루나,1,2
2,7083203,초간단 고추잡채,https://www.10000recipe.com/recipe/7083203,3,3,라따또이,1,3
3,7083202,상큼새콤달콤 사과 오이 샐러드,https://www.10000recipe.com/recipe/7083202,5,5,뿡씨스터즈,1,4
4,7083201,맛보장 오이참치마요샌드위치 만들기,https://www.10000recipe.com/recipe/7083201,5,5,레몬콩,1,5
5,7083200,명절 숙주나물 아삭하게 무치는 법 숙주나물 무침 레시피,https://www.10000recipe.com/recipe/7083200,0,0,에궁이궁,1,6
6,7083199,먹고 또 생각나는 진한 소스가 매력적인 돼지고기 미트소스 스파게티 레시피,https://www.10000recipe.com/recipe/7083199,1,1,에궁이궁,1,7
7,7083197,"열무김치 담그는법, 아삭하고 시원한 열무김치 레시피",https://www.10000recipe.com/recipe/7083197,0,0,얌얌쿡,1,8
8,7083196,목살채소찜 많이 먹어도 속 편하고 부담 없는 돼지고기찜,https://www.10000recipe.com/recipe/7083196,8,8,아임레시피,1,9
9,7083195,소고기 주먹밥 만들기 한그릇 유아식 레시피,https://www.10000recipe.com/recipe/7083195,4,4,보통엄마밥상,1,10


In [75]:
def get_total_recipe_count():

    response = session.get(
        LIST_URL,
        params={
            "order": "date",
            "page": 1
        },
        timeout=20
    )

    response.raise_for_status()

    # 만개의레시피는 UTF-8
    response.encoding = "utf-8"

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # 페이지 전체 텍스트로 변환
    # 태그 사이에 나뉜 텍스트도 하나로 합쳐짐
    text = soup.get_text(
        " ",
        strip=True
    )

    # 굳이 뒤 문장까지 맞추지 않고
    # "총 숫자" 부분만 찾음
    match = re.search(
        r"총\s*([\d,]+)",
        text
    )

    if not match:

        print("status_code:", response.status_code)
        print("URL:", response.url)
        print("HTML 길이:", len(response.text))

        # 디버깅용
        print(text[:1000])

        raise ValueError(
            "전체 레시피 개수를 찾지 못했습니다."
        )

    total_count = int(
        match.group(1).replace(",", "")
    )

    return total_count

In [76]:
total_count = get_total_recipe_count()

print(total_count)
print(f"{total_count:,}개")

276554
276,554개


In [77]:
first_page = parse_list_page(1)

page_size = len(first_page)

print(
    "페이지당 레시피:",
    page_size
)

total_pages = math.ceil(
    total_count / page_size
)

print(
    "예상 전체 페이지:",
    total_pages
)

페이지당 레시피: 40
예상 전체 페이지: 6914


In [78]:
all_rows = []

for page in tqdm(
    range(1, total_pages + 1)
):

    try:
        rows = parse_list_page(page)

        if not rows:
            print(
                f"{page} 페이지 데이터 없음"
            )
            continue

        all_rows.extend(rows)

        # 50페이지마다 백업
        if page % 50 == 0:

            temp_df = pd.DataFrame(
                all_rows
            )

            temp_df = (
                temp_df
                .drop_duplicates(
                    subset="recipe_id"
                )
            )

            temp_df.to_csv(
                LIST_FILE,
                index=False,
                encoding="utf-8-sig"
            )

        # 서버에 너무 빠르게 요청하지 않기
        time.sleep(
            random.uniform(
                0.8,
                1.39
            )
        )

    except Exception as e:

        print(
            f"페이지 {page} 오류:",
            e
        )

  0%|          | 1/6914 [00:01<3:07:17,  1.63s/it]Exception ignored in: <function tqdm.__del__ at 0x0000014054231BC0>
Traceback (most recent call last):
  File "c:\Users\Playdata\Desktop\mle-01-p2-team2\.venv\Lib\site-packages\tqdm\std.py", line 1154, in __del__
    self.close()
  File "c:\Users\Playdata\Desktop\mle-01-p2-team2\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
100%|██████████| 6914/6914 [2:47:33<00:00,  1.45s/it]  


In [79]:
recipe_df = pd.DataFrame(
    all_rows
)

recipe_df = (
    recipe_df
    .drop_duplicates(
        subset="recipe_id"
    )
)

recipe_df.to_csv(
    LIST_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    "수집 레시피:",
    len(recipe_df)
)

수집 레시피: 276550


In [80]:
if LIST_FILE.exists():

    recipe_df = pd.read_csv(
        LIST_FILE,
        dtype={
            "recipe_id": str
        }
    )

    start_page = (
        recipe_df["list_page"].max()
        + 1
    )

    all_rows = recipe_df.to_dict(
        "records"
    )

else:

    start_page = 1
    all_rows = []


print(
    "시작 페이지:",
    start_page
)

시작 페이지: 6915


In [81]:
for page in tqdm(
    range(
        start_page,
        total_pages + 1
    )
):

    try:

        rows = parse_list_page(page)

        all_rows.extend(rows)

        if page % 50 == 0:

            temp_df = pd.DataFrame(
                all_rows
            )

            temp_df = (
                temp_df
                .drop_duplicates(
                    subset="recipe_id"
                )
            )

            temp_df.to_csv(
                LIST_FILE,
                index=False,
                encoding="utf-8-sig"
            )

        time.sleep(
            random.uniform(
                0.8,
                1.3
            )
        )

    except Exception as e:

        print(
            page,
            e
        )

0it [00:00, ?it/s]


In [82]:
recipe_df = pd.read_csv(
    LIST_FILE,
    dtype={
        "recipe_id": str
    }
)

recipe_df["views"] = (
    pd.to_numeric(
        recipe_df["views"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

In [83]:
top_views = (
    recipe_df
    .sort_values(
        "views",
        ascending=False
    )
    .head(7000)
    .copy()
)

top_views["selection_reason"] = (
    "top_views"
)

top_views.head(20)

,recipe_id,title,url,views,views_raw,author,list_page,page_rank,selection_reason
196777,6876357,닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^,https://www.10000recipe.com/recipe/6876357,6199000,619.9만,주부9단요리톡톡,4920,24,top_views
266105,1785098,돼지고기 김치찌개 맛내는 비법,https://www.10000recipe.com/recipe/1785098,5910000,591만,완소연홍,6653,36,top_views
199195,6873683,"엄마의 레시피, 소고기 미역국 끓이는 법",https://www.10000recipe.com/recipe/6873683,5661000,566.1만,베리츄,4981,2,top_views
171767,6903507,"오징어 볶음, 향과 맛이 일품! 백종원 오징어 볶음",https://www.10000recipe.com/recipe/6903507,5414000,541.4만,hancy002,4295,14,top_views
194171,6879215,소불고기 황금 양념 레시피,https://www.10000recipe.com/recipe/6879215,5103000,510.3만,스와티라마,4855,18,top_views
193866,6879533,생생정보통 잡채 황금레시피 이거였네,https://www.10000recipe.com/recipe/6879533,4997000,499.7만,엘린84,4847,33,top_views
205169,6867256,백종원레시피로 만든 콩나물무침으로 밥 한 끼 뚝딱 ~,https://www.10000recipe.com/recipe/6867256,4880000,488만,뽕림이,5130,16,top_views
189738,6883937,닭볶음탕 닭도리탕 황금레시피 짱짱맛 칼칼함이 남달라,https://www.10000recipe.com/recipe/6883937,4639000,463.9만,블레스그레이스,4744,25,top_views
163688,6912220,"순두부찌개. 바지락, 고기 없이도 기가 막힌 순두부찌개 만드는 법 / 만들기 / 순...",https://www.10000recipe.com/recipe/6912220,4443000,444.3만,케이쿡,4093,14,top_views
169679,6905743,절대 실패없는 제육볶음 황금레시피 감칠맛과 매운맛이 좋아요~!!,https://www.10000recipe.com/recipe/6905743,4314000,431.4만,따봉이kitchen,4243,6,top_views


In [84]:
latest_df = (
    recipe_df
    .sort_values(
        [
            "list_page",
            "page_rank"
        ]
    )
)

In [85]:
top_ids = set(
    top_views["recipe_id"]
)

In [86]:
latest_unique = (
    latest_df[
        ~latest_df["recipe_id"].isin(
            top_ids
        )
    ]
    .head(3000)
    .copy()
)

latest_unique[
    "selection_reason"
] = "latest"

In [87]:
selected_df = pd.concat(
    [
        top_views,
        latest_unique
    ],
    ignore_index=True
)

In [88]:
selected_df = (
    selected_df
    .drop_duplicates(
        subset="recipe_id"
    )
)

In [89]:
selected_df = (
    selected_df
    .drop_duplicates(
        subset="recipe_id"
    )
)   

In [90]:
selected_df.to_csv(
    SELECTED_FILE,
    index=False,
    encoding="utf-8-sig"
)

In [91]:
def clean_text(element):

    if element is None:
        return None

    return element.get_text(
        " ",
        strip=True
    )

In [92]:
def parse_recipe_detail(
    recipe_id
):

    url = (
        f"{BASE_URL}/recipe/"
        f"{recipe_id}"
    )

    response = session.get(
        url,
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # ------------------
    # 제목
    # ------------------

    title = clean_text(
        soup.select_one(
            ".view2_summary h3"
        )
    )

    # ------------------
    # 설명
    # ------------------

    description = clean_text(
        soup.select_one(
            ".view2_summary_in"
        )
    )

    # ------------------
    # 인분 / 시간 / 난이도
    # ------------------

    servings = clean_text(
        soup.select_one(
            ".view2_summary_info1"
        )
    )

    cooking_time = clean_text(
        soup.select_one(
            ".view2_summary_info2"
        )
    )

    difficulty = clean_text(
        soup.select_one(
            ".view2_summary_info3"
        )
    )

    # ------------------
    # 재료
    # ------------------

    ingredients = []
    seasonings = []
    tools = []

    ingredient_area = soup.select_one(
        ".ready_ingre3"
    )

    if ingredient_area:

        groups = ingredient_area.select(
            "ul"
        )

        for group in groups:

            group_title_el = (
                group.select_one("b")
            )

            group_title = (
                clean_text(group_title_el)
                or "재료"
            )

            for li in group.select("li"):

                full_text = li.get_text(
                    " ",
                    strip=True
                )

                # 구매 버튼 제거
                full_text = (
                    full_text
                    .replace(
                        "구매",
                        ""
                    )
                    .strip()
                )

                name_el = (
                    li.select_one("a")
                )

                name = (
                    clean_text(name_el)
                    if name_el
                    else None
                )

                amount_el = (
                    li.select_one("span")
                )

                amount = (
                    clean_text(amount_el)
                    if amount_el
                    else None
                )

                if amount:
                    amount = amount.replace(
                        "구매",
                        ""
                    ).strip()

                ingredient = {
                    "group": group_title,
                    "name": name,
                    "amount": amount,
                    "raw": full_text
                }

                # 조리도구
                if "조리도구" in group_title:

                    tools.append(
                        ingredient
                    )

                # 양념
                elif (
                    "양념" in group_title
                    or "소스" in group_title
                ):

                    seasonings.append(
                        ingredient
                    )

                else:

                    ingredients.append(
                        ingredient
                    )

    # ------------------
    # 조리 순서
    # ------------------

    steps = []

    step_elements = soup.select(
        ".view_step_cont"
    )

    # 사이트 구조 변화 대비
    if not step_elements:

        step_elements = soup.select(
            '[id^="stepdescr"]'
        )

    for i, step in enumerate(
        step_elements,
        start=1
    ):

        text = step.get_text(
            " ",
            strip=True
        )

        if text:

            steps.append({
                "order": i,
                "text": text
            })

    # ------------------
    # 등록일 / 수정일
    # ------------------

    whole_text = soup.get_text(
        " ",
        strip=True
    )

    created_match = re.search(
        r"등록일\s*:\s*"
        r"(\d{4}-\d{2}-\d{2})",
        whole_text
    )

    updated_match = re.search(
        r"수정일\s*:\s*"
        r"(\d{4}-\d{2}-\d{2})",
        whole_text
    )

    created_at = (
        created_match.group(1)
        if created_match
        else None
    )

    updated_at = (
        updated_match.group(1)
        if updated_match
        else None
    )

    return {
        "source": "10000recipe",
        "source_id": str(recipe_id),
        "source_url": url,

        "title": title,
        "description": description,

        "servings": servings,
        "cooking_time": cooking_time,
        "difficulty": difficulty,

        "ingredients": ingredients,
        "seasonings": seasonings,
        "tools": tools,

        "steps": steps,

        "created_at": created_at,
        "updated_at": updated_at
    }

In [93]:
test_id = selected_df.iloc[0][
    "recipe_id"
]

test_recipe = parse_recipe_detail(
    test_id
)

test_recipe

{'source': '10000recipe',
 'source_id': '6876357',
 'source_url': 'https://www.10000recipe.com/recipe/6876357',
 'title': '닭볶음탕 진짜진짜 황금레시피 알려 드려요~~^^',
 'description': '닭볶음탕 레시피입니다 아래 유튜브 동영상도 있으니 주소 클릭하셔서 참고 하셔도 조을듯요~~~^^ https://youtu.be/Q3SqbUUuSrY 다른요리가 궁금하시면 채널주소로 ㄱㄱ https://m.youtube.com/channel/UC62lSbFDMublcifIjVexQFQ 저희가족 가끔가는 닭볶음탕 맛집이 있는데요 너무 맛있어서 제가 한번 따라해 봤어요 인터넷도 여기저기 뒤져보고 했는데 딱 요레시피가거의 근접한 맛이 나더라구요 닭볶음탕 좋아하시는분들 꼭 한번 해보시길요~~',
 'servings': '3인분',
 'cooking_time': '60분 이내',
 'difficulty': '초급',
 'ingredients': [{'group': '[재료]',
   'name': '닭',
   'amount': '1마리',
   'raw': '닭 1마리'},
  {'group': '[재료]', 'name': '당면', 'amount': '', 'raw': '당면'},
  {'group': '[재료]', 'name': '감자', 'amount': '3개', 'raw': '감자 3개'},
  {'group': '[재료]', 'name': '당근', 'amount': '1/2쪽', 'raw': '당근 1/2쪽'},
  {'group': '[재료]', 'name': '양파', 'amount': '1~1.5개', 'raw': '양파 1~1.5개'},
  {'group': '[재료]', 'name': '고추', 'amount': '3개', 'raw': '고추 3개'},
  {'group': '[재료]', 'name': '대파', 'amount': '1대', 'raw'

In [94]:
test_recipe = parse_recipe_detail(
    "6876638"
)

test_recipe

{'source': '10000recipe',
 'source_id': '6876638',
 'source_url': 'https://www.10000recipe.com/recipe/6876638',
 'title': '김치 볶음밥 :초간단 한그릇 요리',
 'description': '안녕하세요~ 몽베이향입니다 :^) 항상 냉장고에 있는 반찬~ 김치를 넣어서 초간단으로 한그릇 요리를 했어요~ 오직 김치만 넣어 만든 김치 볶음밥! 한번 만들어 볼까요~?',
 'servings': '1인분',
 'cooking_time': '10분 이내',
 'difficulty': '아무나',
 'ingredients': [{'group': '[재료(1인분)]',
   'name': '밥',
   'amount': '1그릇',
   'raw': '밥 1그릇'},
  {'group': '[재료(1인분)]', 'name': '김치', 'amount': '100g', 'raw': '김치 100g'},
  {'group': '[재료(1인분)]', 'name': '김칫국물', 'amount': '3큰술', 'raw': '김칫국물 3큰술'},
  {'group': '[재료(1인분)]', 'name': '식초', 'amount': '1T', 'raw': '식초 1T'},
  {'group': '[재료(1인분)]', 'name': '설탕', 'amount': '1/3T', 'raw': '설탕 1/3T'},
  {'group': '[재료(1인분)]', 'name': '간장', 'amount': '1/2~1T', 'raw': '간장 1/2~1T'},
  {'group': '[재료(1인분)]',
   'name': '참기름',
   'amount': '1/2~1T',
   'raw': '참기름 1/2~1T'},
  {'group': '[재료(1인분)]', 'name': '고추기름', 'amount': '1T', 'raw': '고추기름 1T'},
  {'group': '[재료(1인분)]', 'na

In [95]:
print(
    test_recipe["title"]
)

print(
    test_recipe["servings"]
)

print(
    test_recipe["cooking_time"]
)

print("\n재료")

for item in test_recipe[
    "ingredients"
]:
    print(item)

print("\n양념")

for item in test_recipe[
    "seasonings"
]:
    print(item)

print("\n조리도구")

for item in test_recipe[
    "tools"
]:
    print(item)

print("\n조리순서")

for step in test_recipe[
    "steps"
]:
    print(
        step["order"],
        step["text"]
    )

김치 볶음밥 :초간단 한그릇 요리
1인분
10분 이내

재료
{'group': '[재료(1인분)]', 'name': '밥', 'amount': '1그릇', 'raw': '밥 1그릇'}
{'group': '[재료(1인분)]', 'name': '김치', 'amount': '100g', 'raw': '김치 100g'}
{'group': '[재료(1인분)]', 'name': '김칫국물', 'amount': '3큰술', 'raw': '김칫국물 3큰술'}
{'group': '[재료(1인분)]', 'name': '식초', 'amount': '1T', 'raw': '식초 1T'}
{'group': '[재료(1인분)]', 'name': '설탕', 'amount': '1/3T', 'raw': '설탕 1/3T'}
{'group': '[재료(1인분)]', 'name': '간장', 'amount': '1/2~1T', 'raw': '간장 1/2~1T'}
{'group': '[재료(1인분)]', 'name': '참기름', 'amount': '1/2~1T', 'raw': '참기름 1/2~1T'}
{'group': '[재료(1인분)]', 'name': '고추기름', 'amount': '1T', 'raw': '고추기름 1T'}
{'group': '[재료(1인분)]', 'name': '깨', 'amount': '조금', 'raw': '깨 조금'}

양념

조리도구

조리순서
1 김치, 김칫국물에 설탕, 식초를 넣고 섞은 후, 가위로 다지듯 작게 잘라 주세요
2 달군 팬에 고추 기름, 잘게 잘라 둔 김치를 넣고 중불에서 김치가 살짝 익을 정도로 볶아 주세요
3 센불로 밥을 넣고 볶아 주세요
4 밥이 없는 부분에 간장을 뿌린 후 밥과 잘 볶아 주세요 (센불로 밥의 수분을 살짝 없애는 느낌이 될때 까지)
5 불을 끈 상태에서 참기름을 뿌려서 잔열로 볶아 주세요
6 그릇에 담아서 접시로 옮겨 주시면 돼요~ (달걀이나 이런저런 데코를 할 경우)


In [96]:
selected_df = pd.read_csv(
    SELECTED_FILE,
    dtype={
        "recipe_id": str
    }
)

In [97]:
completed_ids = set()

if DETAIL_FILE.exists():

    with open(
        DETAIL_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:

                row = json.loads(
                    line
                )

                completed_ids.add(
                    str(
                        row["source_id"]
                    )
                )

            except:
                pass


print(
    "이미 완료:",
    len(completed_ids)
)

이미 완료: 0


In [98]:
with open(
    DETAIL_FILE,
    "a",
    encoding="utf-8"
) as f:

    for row in tqdm(
        selected_df.itertuples(),
        total=len(selected_df)
    ):

        recipe_id = str(
            row.recipe_id
        )

        # 이미 수집한 건 넘어가기
        if recipe_id in completed_ids:
            continue

        try:

            recipe = (
                parse_recipe_detail(
                    recipe_id
                )
            )

            recipe[
                "views"
            ] = int(row.views)

            recipe[
                "selection_reason"
            ] = row.selection_reason

            f.write(
                json.dumps(
                    recipe,
                    ensure_ascii=False
                )
                + "\n"
            )

            f.flush()

            completed_ids.add(
                recipe_id
            )

        except Exception as e:

            print(
                f"{recipe_id} 오류:",
                e
            )

        time.sleep(
            random.uniform(
                1.0,
                1.5
            )
        )

100%|██████████| 10000/10000 [5:02:05<00:00,  1.81s/it] 
